<a href="https://colab.research.google.com/github/VukasinA/ML_projekti/blob/main/MLDom2Zad2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zadatak 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Preprocesiranje podataka
def preprocess_data(df):
    df = df.copy()

    # Popunjavanje nedostajućih vrednosti
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna('S')

    # Konverzija kategorijskih promenljivih
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

    # Kreiranje novih feature-a
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # Normalizacija
    for col in ['Age', 'Fare']:
        df[col] = (df[col] - df[col].mean()) / df[col].std()

    return df[['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']]

# Priprema podataka
X_train = preprocess_data(train_df).values
y_train = train_df['Survived'].values.reshape(-1, 1)
X_test = preprocess_data(test_df).values

indices = np.random.permutation(len(X_train))
train_idx, val_idx = indices[:int(0.8*len(X_train))], indices[int(0.8*len(X_train)):]
X_val, y_val = X_train[val_idx], y_train[val_idx]
X_train, y_train = X_train[train_idx], y_train[train_idx]

# Aktivacione funkcije ReLU i Softmax
def relu(Z):
    return np.maximum(0, Z)

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

# Inicijalizacija parametara
def initialize_parameters(layer_dims):
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2./layer_dims[l-1])
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

# Forward propagacija
def forward_propagation(X, parameters):
    caches = []
    A = X.T  # Transponujemo za pravilne dimenzije (broj featurea x samples)
    L = len(parameters) // 2

    for l in range(1, L):
        A_prev = A
        W = parameters['W' + str(l)]
        b = parameters['b' + str(l)]
        Z = np.dot(W, A_prev) + b
        A = relu(Z)
        caches.append((A_prev, W, b, Z))

    # Izlazni sloj
    W = parameters['W' + str(L)]
    b = parameters['b' + str(L)]
    Z = np.dot(W, A) + b
    AL = softmax(Z)
    caches.append((A, W, b, Z))

    return AL, caches

# Funkcija troska
def compute_cost(AL, Y):
    m = Y.shape[0]
    cost = -np.sum(Y.T * np.log(AL + 1e-8)) / m
    return np.squeeze(cost)
    #cross entropy loss

# Backward propagacija
def backward_propagation(AL, Y, caches, parameters):
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    Y = Y.T

    # Izlazni sloj
    dZ = AL - Y
    current_cache = caches[L-1]
    A_prev, W, b, Z = current_cache
    grads['dW' + str(L)] = np.dot(dZ, A_prev.T) / m
    grads['db' + str(L)] = np.sum(dZ, axis=1, keepdims=True) / m

    # Propagacija unazad
    for l in reversed(range(L-1)):
        current_cache = caches[l]
        A_prev, W, b, Z = current_cache
        dA = np.dot(parameters['W' + str(l+2)].T, dZ)
        dZ = dA * (Z > 0).astype(float)
        grads['dW' + str(l+1)] = np.dot(dZ, A_prev.T) / m
        grads['db' + str(l+1)] = np.sum(dZ, axis=1, keepdims=True) / m

    return grads

def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2
    for l in range(1, L+1):
        parameters['W' + str(l)] -= learning_rate * grads['dW' + str(l)]
        parameters['b' + str(l)] -= learning_rate * grads['db' + str(l)]
    return parameters
    #gradient descent

def prepare_targets(y):
    y_onehot = np.zeros((y.shape[0], 2))
    y_onehot[np.arange(y.shape[0]), y.flatten()] = 1
    return y_onehot

# Glavna trening funkcija
def train_model(X, Y, layers_dims, learning_rate=0.01, epochs=100, batch_size=32):
    parameters = initialize_parameters(layers_dims)
    costs = []

    y_onehot = prepare_targets(Y)

    for i in range(epochs):
        indices = np.random.permutation(X.shape[0])
        X_shuffled = X[indices]
        y_shuffled = y_onehot[indices]

        for j in range(0, X.shape[0], batch_size):
            X_batch = X_shuffled[j:j+batch_size]
            y_batch = y_shuffled[j:j+batch_size]

            # Forward propag
            AL, caches = forward_propagation(X_batch, parameters)
            # Trosak
            cost = compute_cost(AL, y_batch)
            # Backward propag
            grads = backward_propagation(AL, y_batch, caches, parameters)
            parameters = update_parameters(parameters, grads, learning_rate)

        if i % 10 == 0:
            costs.append(cost)
            print(f"Epoha {i}, Trosak: {cost:.4f}")

    return parameters, costs

# Pokretanje treninga
layer_dims = [X_train.shape[1], 32, 16, 2]  # 7 ulaza, 2 izlaza
parameters, costs = train_model(X_train, y_train, layer_dims, learning_rate=0.01, epochs=100, batch_size=32)

# Funkcija za predikciju
def predict(X, parameters):
    AL, _ = forward_propagation(X, parameters)
    predictions = np.argmax(AL, axis=0)
    return predictions

train_pred = predict(X_train, parameters)
val_pred = predict(X_val, parameters)

print(f"Trening acc: {100 * np.mean(train_pred == y_train.flatten()):.3f} %")
print(f"Validacija acc: {100 * np.mean(val_pred == y_val.flatten()):.3f} %")


Epoha 0, Trosak: 0.7525
Epoha 10, Trosak: 0.4475
Epoha 20, Trosak: 0.2934
Epoha 30, Trosak: 0.2125
Epoha 40, Trosak: 0.3082
Epoha 50, Trosak: 0.2320
Epoha 60, Trosak: 0.6065
Epoha 70, Trosak: 0.6034
Epoha 80, Trosak: 0.2212
Epoha 90, Trosak: 0.5800
Trening acc: 82.30 %
Validacija acc: 79.33 %
